# GSB 5544 — NumPy Essentials: The 7 Concepts Behind Every DataFrame  
**SOLUTION VERSION**

Pandas is built on **NumPy**. Every column of a DataFrame is a NumPy array underneath, so understanding 7 NumPy ideas makes pandas make sense. As before, each concept answers a **question**.

**Data set:** the same `coffee_purchases.csv`. We'll pull columns *out* of pandas into NumPy to see what's really there.

In [1]:
import numpy as np
import pandas as pd

coffee = pd.read_csv("https://raw.githubusercontent.com/gato365/gsb5544_instructor_learn_prep/main/assignments/Data/coffee_purchases.csv")   # <-- put the CSV URL / path here
coffee["spend"] = coffee["amount"].abs()
coffee.head(3)

,date,description,amount,account,month,year,day_of_week,spend
0,17826,SQ *LUCY'S COFFEE C 10/21 PURCHASE SAN LUIS OB...,-20.10,Kelley,October,2018,Monday,20.10
1,17833,SQ *LUCY'S COFFEE C 10/27 PURCHASE SAN LUIS OB...,-7.43,Eman,October,2018,Monday,7.43
2,18050,SQ *LUCY'S COFFEE CO 06/02 PURCHASE San Luis O...,-2.75,Eman,June,2019,Monday,2.75


---
## Concept 1 — The `ndarray`: *What is a pandas column made of?*

A NumPy array is a grid of values that are **all the same type**. That single rule is what makes it fast. A Series is an array plus an index.

In [2]:
spend = coffee["spend"].to_numpy()      # pull the raw array out of the Series
print(type(spend))
print(spend[:10])

<class 'numpy.ndarray'>
[20.1   7.43  2.75  2.75  9.21  5.33 12.91  2.75  2.75  3.25]


In [3]:
# Q: What do I need to know about an array?  shape, ndim, dtype, size
print("shape:", spend.shape)
print("ndim :", spend.ndim)
print("dtype:", spend.dtype)
print("size :", spend.size)

shape: (821,)
ndim : 1
dtype: float64
size : 821


In [4]:
# Building arrays from scratch
print(np.array([1, 2, 3]))
print(np.zeros(4))
print(np.arange(0, 10, 2))        # start, stop (exclusive), step
print(np.linspace(0, 1, 5))       # 5 evenly spaced numbers from 0 to 1

[1 2 3]
[0. 0. 0. 0.]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]


---
## Concept 2 — dtype: *Why does one text value change a whole column?*

An array has **one** dtype. Mixing types forces NumPy to pick the most general one (usually a string), which is why a single `"N/A"` in a CSV turns a numeric column into `object`.

In [5]:
print(np.array([1, 2, 3]).dtype)
print(np.array([1, 2.5, 3]).dtype)       # int gets promoted to float
print(np.array([1, 2, "three"]).dtype)   # everything becomes a string!

int64
float64
<U21


In [6]:
# Q: How do I convert?  .astype()  (same idea as pandas)
years = coffee["year"].to_numpy()
print(years.dtype)
print(years.astype(str)[:5])
print(years.astype(float)[:5])

int64
['2018' '2018' '2019' '2019' '2019']
[2018. 2018. 2019. 2019. 2019.]


---
## Concept 3 — Vectorization: *How do I do math on 821 numbers without a loop?*

Operations apply to **every element at once**. This is the reason `coffee["amount"] * 2` works in pandas. No `for` loop needed — and it's ~100× faster.

In [7]:
# Q: What would each purchase cost with 8% tax?
with_tax = spend * 1.08
with_tax[:5]

array([21.708 ,  8.0244,  2.97  ,  2.97  ,  9.9468])

In [8]:
# Array + array works element by element (shapes must match)
amount = coffee["amount"].to_numpy()
print((amount + spend)[:5])              # negative + positive = 0 for every row

[0. 0. 0. 0. 0.]


In [9]:
# Math functions are vectorized too
print(np.round(spend, 0)[:5])
print(np.log(spend)[:5])
print(np.sqrt(spend)[:5])

[20.  7.  3.  3.  9.]
[3.00071982 2.00552586 1.01160091 1.01160091 2.22028985]
[4.48330235 2.72580263 1.6583124  1.6583124  3.03479818]


In [10]:
# Q: How much faster than a loop?
big = np.random.rand(1_000_000)

%timeit [x * 1.08 for x in big]
%timeit big * 1.08

47.9 ms per loop
0.17 ms per loop


---
## Concept 4 — Aggregation: *How do I summarize a column in one number?*

`.mean()`, `.sum()`, `.min()`, `.max()`, `.std()` on a Series are NumPy aggregations. Also learn `argmax`/`argmin` — *where* the max is — which is the idea behind pandas' `idxmax`.

In [11]:
# Q: What's the total, average, and biggest coffee purchase?
print("total :", spend.sum().round(2))
print("mean  :", spend.mean().round(2))
print("median:", np.median(spend))
print("max   :", spend.max())
print("std   :", spend.std().round(2))

total : 4974.47
mean  : 6.06
median: 4.75
max   : 55.89
std   : 5.3


In [12]:
# Q: WHICH purchase was the biggest?  argmax gives the position
i = spend.argmax()
print(i)
coffee.iloc[i]

703


date                                                       20094
description    SQ *SCOUT COFFEE 01/04 MOBILE PURCHASE San Lui...
amount                                                    -55.89
account                                                     Eman
month                                                    January
year                                                        2025
day_of_week                                               Monday
spend                                                      55.89
Name: 703, dtype: object

In [13]:
# Q: What are the quartiles?  (pd.qcut used these under the hood)
np.percentile(spend, [25, 50, 75])

array([3.25, 4.75, 7.25])

---
## Concept 5 — Boolean masks: *How do I pick out the rows that meet a condition?*

A comparison on an array returns an array of `True`/`False` — a **mask**. Indexing with a mask keeps the `True` rows. This is exactly what `coffee[coffee["amount"] < -10]` does in pandas.

In [14]:
# Q: Which purchases were over $10?
mask = spend > 10
print(mask[:10])
print("how many True:", mask.sum())        # True counts as 1
print("fraction     :", mask.mean().round(3))

[ True False False False False False  True False False False]
how many True: 83
fraction     : 0.101


In [15]:
spend[mask][:10]           # keep only the True positions

array([20.1 , 12.91, 10.5 , 11.61, 24.2 , 12.6 , 16.8 , 13.65, 10.8 ,
       12.29])

In [16]:
# Combine conditions: & (and), | (or), ~ (not). Parentheses required.
dow = coffee["day_of_week"].to_numpy()
weekend_big = (spend > 10) & ((dow == "Saturday") | (dow == "Sunday"))
coffee[weekend_big]

,date,description,amount,account,month,year,day_of_week,spend
481,19315,SQ *KRAKEN AVILA Avila Beach CA Digital Accoun...,-11.00,Credit Card,November,2022,Saturday,11.00
482,19315,SQ *KRAKEN AVILA Avila Beach CA Digital Accoun...,-11.00,Credit Card,November,2022,Saturday,11.00
560,19469,SQ *SCOUT COFFEE San Luis Obis CA null XXXXXXX...,-10.70,Credit Card,April,2023,Saturday,10.70
562,19470,TST* KRAKEN - AVILA AVILA BEACH CA null XXXXXX...,-12.00,Credit Card,April,2023,Sunday,12.00
564,19477,TST* KRAKEN - AVILA AVILA BEACH CA null XXXXXX...,-17.50,Credit Card,April,2023,Sunday,17.50
683,19883,SQ *SCOUT COFFEE San Luis Obis CA null XXXXXXX...,-32.24,Credit Card,June,2024,Sunday,32.24
760,20008,SQ *FIELD DAY COFFEE SAN LUIS OBIS CA null XXX...,-13.05,Credit Card,October,2024,Saturday,13.05


In [17]:
# Q: Where (which positions) are the True values?
np.where(spend > 15)[0]

array([  0,  65,  79, 176, 179, 188, 413, 428, 564, 608, 623, 625, 683,
       697, 703, 725, 727, 749, 818])

In [18]:
# np.where as an if/else on every element  -> a new categorical variable
size = np.where(spend > 10, "large", "small")
pd.Series(size).value_counts()

small    738
large     83
Name: count, dtype: int64

---
## Concept 6 — Indexing & slicing 2-D arrays: *Where do `.iloc[rows, cols]` rules come from?*

A DataFrame's numbers are a 2-D array. `arr[rows, cols]` with integers and `:` slices is exactly `.iloc`. Position-based, end-exclusive.

In [19]:
nums = coffee[["amount", "year", "spend"]].to_numpy()
print(nums.shape)
nums[:3]

(821, 3)


array([[ -20.1 , 2018.  ,   20.1 ],
       [  -7.43, 2018.  ,    7.43],
       [  -2.75, 2019.  ,    2.75]])

In [20]:
print(nums[0, 2])          # row 0, column 2  (like .iloc[0, 2])
print(nums[0:3, 0])        # rows 0-2, column 0
print(nums[:, 1][:5])      # all rows, column 1
print(nums[-2:, :])        # last two rows, all columns

20.1
[-20.1   -7.43  -2.75]
[2018. 2018. 2019. 2019. 2019.]
[[  -4.95 2025.      4.95]
 [  -6.5  2025.      6.5 ]]


In [21]:
# Reshaping: the same 12 numbers as 3x4 or 4x3
a = np.arange(12)
print(a.reshape(3, 4))
print(a.reshape(4, 3).T)   # .T = transpose

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[[ 0  3  6  9]
 [ 1  4  7 10]
 [ 2  5  8 11]]


---
## Concept 7 — Missing values (`NaN`): *Why does my average come out as `nan`?*

`np.nan` is a float that means "missing". Plain NumPy math **propagates** it; pandas methods **skip** it. Knowing this explains why `Series.mean()` and `np.mean()` can disagree.

In [22]:
x = np.array([4.5, np.nan, 3.0, 7.25])

print(np.mean(x))          # nan  -> one missing value poisons the result
print(np.nanmean(x))       # 4.9166 -> ignore the missing value
print(pd.Series(x).mean()) # pandas skips NaN by default

nan
4.916666666666667
4.916666666666667


In [23]:
# Q: How do I find missing values?  (nan != nan, so use isnan)
print(x == np.nan)         # all False -- the trap
print(np.isnan(x))         # the right way
print(np.isnan(x).sum(), "missing")

[False False False False]
[False  True False False]
1 missing


In [24]:
# Q: Does the coffee data have any missing values?
coffee.isna().sum()

date           0
description    0
amount         0
account        0
month          0
year           0
day_of_week    0
spend          0
dtype: int64

In [25]:
# Filling or dropping
print(np.nan_to_num(x, nan=0))
print(x[~np.isnan(x)])

[4.5  0.   3.   7.25]
[4.5  3.   7.25]


---
## Summary — 7 NumPy concepts and the pandas feature they explain

| # | NumPy concept | Question | pandas feature it explains |
|---|---|---|---|
| 1 | `ndarray` (shape, ndim, dtype) | What is a column made of? | Series = array + index |
| 2 | one dtype per array | Why does one text value change a column? | `object` columns, `.astype()` |
| 3 | vectorization | Math on 821 values without a loop? | `df["a"] * 2`, `df["a"] + df["b"]` |
| 4 | aggregation, `argmax` | Summarize in one number? Where is the max? | `.mean()`, `.sum()`, `.idxmax()` |
| 5 | boolean masks, `np.where` | Pick rows that meet a condition? | `df[df["a"] > 10]`, `&`, `\|`, `~` |
| 6 | 2-D indexing `[rows, cols]` | Where do `.iloc` rules come from? | `.iloc[r, c]`, end-exclusive slices |
| 7 | `NaN` | Why is my mean `nan`? | `.isna()`, `skipna=True`, `.fillna()` |